# Experiment Tracking

![Status](https://img.shields.io/static/v1.svg?label=Status&message=Finished&color=brightgreen)
[![Source](https://img.shields.io/static/v1.svg?label=GitHub&message=Source&color=181717&logo=GitHub)](https://github.com/particle1331/ok-transformer/blob/master/docs/nb/mlops/03-mlflow.ipynb)
[![Stars](https://img.shields.io/github/stars/particle1331/ok-transformer?style=social)](https://github.com/particle1331/ok-transformer)

---

## Introduction

In this module, we will look **experiment tracking** and **model management** using [MLflow](https://mlflow.org/). Manual experiment tracking and model management is error prone, unstandardized, and has low visibility. This makes it difficult for collaboration and reproducing past results. MLflow provides a framework for tracking experiment metadata in a DB and storing artifacts such as serialized (trained) models as well as environment specifications in a remote file store. These are important for reproducing our experiments and performing remote inference. MLflow also has features for managing which models should go in and out of production.

In [ ]:
!mlflow --version

## MLflow on localhost 

Running the MLflow server on [localhost](https://mlflow.org/docs/latest/tracking.html#scenario-3-mlflow-on-localhost-with-tracking-server) port 5001 with a remote artifact store on S3:

```bash
$ mlflow server -h 127.0.0.1 -p 5001 \
    --backend-store-uri=sqlite:///mlflow.db \
    --default-artifact-root=s3://mlflow-artifact-store-3000
    
[2023-06-04 19:03:37 +0800] [38178] [INFO] Starting gunicorn 20.1.0
[2023-06-04 19:03:37 +0800] [38178] [INFO] Listening at: http://127.0.0.1:5001 (38178)
[2023-06-04 19:03:37 +0800] [38178] [INFO] Using worker: sync
[2023-06-04 19:03:37 +0800] [38179] [INFO] Booting worker with pid: 38179
[2023-06-04 19:03:37 +0800] [38180] [INFO] Booting worker with pid: 38180
[2023-06-04 19:03:37 +0800] [38181] [INFO] Booting worker with pid: 38181
[2023-06-04 19:03:37 +0800] [38182] [INFO] Booting worker with pid: 38182
```

```{figure} https://mlflow.org/docs/latest/_images/scenario_3.png
---
width: 80%
---
Our local setup with [SQLite backend](https://github.com/particle1331/ride-duration-prediction/blob/mlflow/ride_duration/experiment/mlflow.db). But with a remote artifact store on AWS S3.
```

**Remark.** The artifact store (i.e. storage of objects produced in experiment runs) can be a directory in the local file system. However, remote storage on S3 would be convenient for loading trained models in remote environments. An S3 bucket in AWS can be easily created using:

```
aws s3api create-bucket --bucket mlflow-artifact-store-3000
```

We will use [code](https://github.com/particle1331/ride-duration-prediction/tree/mlflow) and [data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) from the [previous module](01-intro/modeling-ride-duration) in our experiments.

In [ ]:
!pip install -U git+https://github.com/particle1331/ride-duration-prediction.git@mlflow --force-reinstall > /dev/null

**Remark.** Our experiment scripts are in the [`ride_duration.experiments`](https://github.com/particle1331/ride-duration-prediction/tree/mlflow/ride_duration/experiment) module.

```{figure} ../../img/mlops/03-mlflow.png
---
---
MLflow UI on `localhost:5001`. An experiment is automatically created using `mlflow.set_experiment()` for the first run (see below).
```

## Experiment tracking

### Training scripts

The utils script consists of boilerplate for setting up the **runs**. It also contains utilities around feature engineering and logging. This standardizes data and feature engineering so that the results are comparable. Here `feature_pipe` takes in a sequence of transformations that is applied left to right on the data. The resulting data frame is converted to a list of dictionary features which are then vectorized.

In [ ]:
# experiment/utils.py
!pygmentize -g ~/opt/miniconda3/envs/mlops/lib/python3.9/site-packages/ride_duration/experiment/utils.py

As in the previous notebook, we train on one month and predict on the next. The following runs on our **baseline** linear regression model. This should reproduce results from the previous notebooks. All code inside the context forms a single run:

In [ ]:
# experiment/linear.py
!pygmentize -g ~/opt/miniconda3/envs/mlops/lib/python3.9/site-packages/ride_duration/experiment/linear.py

The next run modifies the above script with some feature engineering. This can be done by simply passing transformations (i.e. `f: df -> df`) as argument to the `feature_pipeline`:

```python
with mlflow.start_run():
    ...
    
    # Feature engineering + selection
    transforms = [add_pudo_column, feature_selector]

    # Fit feature pipe
    feature_pipe = feature_pipeline(transforms)
    ...
```

The complete script is as follows:

In [ ]:
# experiment/linear_pudo.py
!pygmentize -g ~/opt/miniconda3/envs/mlops/lib/python3.9/site-packages/ride_duration/experiment/linear_pudo.py

````{margin}
💡 **Debug flag** when prototyping!
```python
int(os.environ["DEBUG"])
```
````

After running this script, we see the runs register in the UI with the logged data. It is also able to obtain extra metadata such as the `git` commit hash. So best practice is to always commit before running experiments. This may require running smaller versions of your scripts when prototyping (e.g. `export DEBUG=1` in our implementation). You can also store the script as an artifact.

```{figure} ../../img/mlops/03-runs.png
---
---
Our two runs in the MLflow UI. Clicking on `rmse_valid` column sorts the runs based on validation RMSE.
```

```{figure} ../../img/mlops/03-run-details.png
---
---
Details of one run. Default parameters are logged.
```

### Autologging

In this section, we show how to iterate over different models. This really just involves wrapping the run function in a loop. We also enable **autologging** for scikit-learn models. Automatic logging allows you to log metrics, parameters, and models without the need for explicit log statements.

In [ ]:
# experiment/sklearn_trees.py
!pygmentize -g ~/opt/miniconda3/envs/mlops/lib/python3.9/site-packages/ride_duration/experiment/sklearn_trees.py

`RandomForestRegressor` has the best validation score:

```{figure} ../../img/mlops/03-ensembles.png
---
---
```

Notice an increase in parameter count which were automatically logged. Autologging also creates the `MLmodel` which we look at shortly (also notice Models column). It also creates config files such as `conda.yaml` which describe the environment used to run the scripts. This indicates the expected input and output for the model and allows easy loading as indicated:

```{figure} ../../img/mlops/03-skautolog.png
---
---
```

### HPO with XGBoost

In this section, we perform hyperparameter optimization (HPO) on XGBoost using Optuna. The parameters are sampled sequentially using the [TPE algorithm](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html) over the search space defined in the `params` dictionary. This algorithm tends to be more cost-effective than grid and random search as shown in  [Appendix: TPE](03-mlflow-experiment/appendix/TPE) below. Each run corresponds to a choice of hyperparameters which corresponds to a `trial` in the Optuna framework.

In [ ]:
# experiment/xgboost_optuna.py
!pygmentize -g ~/opt/miniconda3/envs/mlops/lib/python3.9/site-packages/ride_duration/experiment/xgboost_optuna.py

Note that we use a flag `pudo` on whether to use the interaction feature between endpoint location IDs instead of each location considered separately. This is part of trial parameters so that more runs will be allocated to the setting works better with other parameters. XGBoost runs can be filtered out using `tags.model = 'xgboost'`:

```{figure} ../../img/mlops/03-xgboost-runs.png
---
---
Runs on the XGBoost model with autologging.
```

```{figure} ../../img/mlops/03-xgboost-intermediate.png
---
---
Comparing validation RMSE curves for 10 runs of the XGBoost model.
```

Below we look at visualizations of hyperparameter interaction:

```{figure} ../../img/mlops/03-pudo-boxplot.png
---
---
Analyzing the effect of the feature engineering step of using the `PU_DO` column only on model performance instead of `PULocationID` and `DOLocationID` separately. Seems like not using it is better. This is a tree-based model after all.
```

```{figure} ../../img/mlops/03-parallel.png
---
---
Filtering out high-performing settings in the parallel coordinates plot.
```

```{figure} ../../img/mlops/03-depth-mcw.png
---
---
Contour plot to visualize interaction of `max_depth` and `min_child_weight`.
```

## Model management

```{margin}
[MLflow models](https://www.mlflow.org/docs/latest/models.html)
```
Recall autologging generates an `MLmodel` file as well as files for approximating the environment used to generate the models. The unique `run_id` that corresponds to a directory in the artifacts store is assigned to each run allows for easily loading these models in remote environments. Note that even code for training these models can optionally be added as artifacts. See [docs](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html).

```{figure} ../../img/mlops/03-mlmodel.png
---
name: 03-mlmodel
---
MLflow model information for the XGBoost model. Note the scikit-learn based feature pipeline also has its own section. Moreover, observe that MLflow provides a uniform API for loading these models using the `mlflow.pyfunc.load_model` function.
```

### Remote model loading

Note that a path to the model is provided [above](03-mlmodel). Loading the model from S3:

In [ ]:
import mlflow

MODEL_URI = "s3://mlflow-artifact-store-3000/1/7b9ea2a58957491b92a42d8ec593f8fc/artifacts/model"
mlflow.pyfunc.load_model(MODEL_URI)

Folder structure in S3 is as shown in the [figure](03-mlmodel):

In [ ]:
!aws s3 ls s3://mlflow-artifact-store-3000/1/7b9ea2a58957491b92a42d8ec593f8fc/artifacts/ --recursive --human-readable

The feature pipeline can therefore be loaded similarly:

In [ ]:
ARTIFACTS_STORE  = "s3://mlflow-artifact-store-3000"
EXPERIMENT_ID    = "1"
RUN_ID           = "7b9ea2a58957491b92a42d8ec593f8fc"
ARTIFACTS_PATH   = f"{ARTIFACTS_STORE}/{EXPERIMENT_ID}/{RUN_ID}/artifacts"
MODEL_URI        = f"{ARTIFACTS_PATH}/model"
FEATURE_PIPE_URI = f"{ARTIFACTS_PATH}/feature_pipe"

# Loading models from S3
model = mlflow.pyfunc.load_model(MODEL_URI)
feature_pipe = mlflow.sklearn.load_model(FEATURE_PIPE_URI)

The models can then be used to make inference:

In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error
from ride_duration.processing import preprocess

# Load data from some data source
data = pd.read_parquet("data/green_tripdata_2021-02.parquet")

# Inference on validation data
X, y = preprocess(data, target=True, filter_target=True)
X = feature_pipe.transform(X)
y_pred = model.predict(X)
print(mean_squared_error(y, y_pred) ** 0.5) # Expected: ~6.108

**Remark.** The same code can be used to make inference using any other model trained in the `nyc-green-taxi` experiment (i.e. `pyfunc` models with a `.predict()` method). Note that feature engineering steps are baked into the `feature_pipe` model so that this script is fairly general.

### Model registry

Consider the scenario where a member of our team chooses a new model for production. As deployment engineers, we naturally have the following questions in mind: What has changed in this new model? Is there any preprocessing needed? What are the dependencies? Without experiment tracking, this requires a lot of back and fort communication. If there is an incident and we had to rollback the model version, we have to manually trace what changed in this new version. Moreover, it might not be possible to get back the previous model as information of how it was trained has been lost. 

Our tracking database and standard model files, solves most of these issues. Having a **model registry** takes care of the last details of model release and staging. If we want to rollback models, we only have to look at the archived models, or earlier versions, which also have their own [model lineage](https://aws.amazon.com/blogs/machine-learning/model-and-data-lineage-in-machine-learning-experimentation/). It is also natural to update models trained on the same task since models typically degrades with time or improve with new techniques so that registering a model with multiple versions that correspond to runs make sense.

```{margin}
[Source: `neptune.ai`](https://i0.wp.com/neptune.ai/wp-content/uploads/2022/10/Model-registry-overview.png?ssl=1)
```
```{figure} https://i0.wp.com/neptune.ai/wp-content/uploads/2022/10/Model-registry-overview.png?ssl=1
---
width: 80%
---
Registering models into stages allows identification for QA and downstream tasks.
```

Suppose we need fast models. We can take the [Pareto front](https://en.wikipedia.org/wiki/Pareto_front) of predict time and validation RMSE. Also taking the nearest model on the right which has better RMSE but is about 10x slower. We imagine this to be an update to the first model.

```{figure} ../../img/mlops/03-pareto-front.png
---
width: 80%
---
Two models at the Pareto front of valid RMSE and predict time.
```

Registering this model can be done by simply clicking the "Register Model" button in the UI. We choose the name `NYCGreenTaxiRideDuration` and we stage the first model in production. The latter model is set to staging. The model registry can be viewed in the Models tab of the UI.

```{figure} ../../img/mlops/03-registered-versions.png
---
---
Registering a model and staging a new version of the model trained on the same task.
```


```{figure} ../../img/mlops/03-register-archive.png
---
---
Transitioning the staged model to production and the current prod model to archived.
```


**Remark.** Note that experiment runs does not always need to involve model training.
It can consist of evaluation (e.g. for large pretrained models) on a variety of tasks or datasets. So model versions here can consist of different versions of a pretrained model or different sizes of an LLM.

## API Workflows

In this section, we look at how to programatically interact with the MLflow server through its client. The idea is that everything that we can do in the UI by clicking buttons, we should be able to do here in code. And the fields that are available using the UI correspond to function arguments of the corresponding API endpoint or client method.

### Experiment tracking

The `MlflowClient` connects to the experiment tracking server. This provides a [CRUD interface](https://en.wikipedia.org/wiki/Create,_read,_update_and_delete) for managing experiments and runs. For example, we can list all experiments:

In [ ]:
from mlflow.tracking import MlflowClient

TRACKING_URI = "http://127.0.0.1:5001"
client = MlflowClient(tracking_uri=TRACKING_URI)

def print_experiment(experiment):
    print(f"(Experiment)")
    print(f"    experiment_id={experiment.experiment_id}")
    print(f"    name='{experiment.name}'")
    print(f"    artifact_location='{experiment.artifact_location}'")
    print()


for experiment in client.search_experiments():
    print_experiment(experiment)

**Remark.** Here we have the SQLite file [`mlflow.db`](https://github.com/particle1331/ride-duration-prediction/blob/mlflow/ride_duration/experiment/mlflow.db) on disk. In practice, you may have a remote tracking database (see [Appendix](03-mlflow-experiment/appendix/AWS-deployment)). But the overall idea is the same &mdash; only the URI changes.

In the previous section, we selected two best performing runs using the UI. This can be done with the client using the `search_runs` method. Note that MLflow stores even deleted experiments. So we specify `ViewType` to `ACTIVE_ONLY` in the search results.

In [ ]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids=1,
    filter_string='metrics.predict_time < 3e-6 and metrics.predict_time < 6.20',
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=15,
    order_by=["metrics.rmse_valid ASC", "metrics.predict_time ASC"]
)

for run in runs:
    name = run.info.run_name
    format_name = name + " " * (16 - len(name)) if len(name) <= 16 else name[:13] + "..."
    print(f"{format_name}  {run.info.run_id}  rmse_valid: {run.data.metrics['rmse_valid']:.3f}  predict_time: {run.data.metrics['predict_time']:.4e}")

**Remark.** This may give us better results than visual inspection when choosing models.

### Model registry

Creating a new model version. Note that if the name does not exist, it registers a new model with the source run as initial version.

In [ ]:
import mlflow

# First run in our above query
RUN_ID = runs[0].info.run_id
MODEL_URI = f"s3://mlflow-artifact-store-3000/1/{RUN_ID}/artifacts/model"

client.create_model_version(
    name="NYCGreenTaxiRideDuration",
    run_id=RUN_ID,
    source=MODEL_URI
);

This version can be transitioned to staging as follows:

In [ ]:
import datetime

def transition_stage(client, model_name, version, stage, archive_existing=False):
    """Transition model version to given stage. Log update in description."""

    client.transition_model_version_stage(
        name=model_name,
        version=version,
        stage=stage,
        archive_existing_versions=archive_existing
    )

    t = datetime.datetime.now()
    s = t.isoformat(timespec="seconds")
    description = client.get_model_version(model_name, version).description or ""

    client.update_model_version(
        name=model_name,
        version=version,
        description=(
            f"[{s}] Model version transitioned to {stage}.\n"
            f"{description}"
        )
    )


transition_stage(client, "NYCGreenTaxiRideDuration", 3, "Staging")

```{figure} ../../img/mlops/03-api-versions.png
---
---
```


```{figure} ../../img/mlops/03-create-version-api.png
---
---
```


The following lists the latest versions for each stage:

In [ ]:
latest_versions = client.get_latest_versions(name="NYCGreenTaxiRideDuration")
for version in latest_versions:
    print(f"Version: {version.version}   Run ID: {version.run_id}   Stage: {version.current_stage}" )

The same transition function can be used to promote v4 to production and archive v3 which is currently in production. Note that we can automatically archive v2 using `archive_existing=True`. But we use the transition function so that the transition is logged in the description.

In [ ]:
transition_stage(client, "NYCGreenTaxiRideDuration", 3, "Production")
transition_stage(client, "NYCGreenTaxiRideDuration", 2, "Archived")

```{figure} ../../img/mlops/03-updated-versions.png
---
---
```

```{figure} ../../img/mlops/03-new-versions.png
---
---
```


### Inference with staged models

Here we load the latest production model from S3:

In [ ]:
MODEL_NAME = "NYCGreenTaxiRideDuration"
STAGE      = "Production"

RUN_ID = client.get_latest_versions(name=MODEL_NAME, stages=[STAGE])[0].run_id

ARTIFACTS_PATH   = f"{ARTIFACTS_STORE}/{EXPERIMENT_ID}/{RUN_ID}/artifacts"
MODEL_URI        = f"{ARTIFACTS_PATH}/model"
FEATURE_PIPE_URI = f"{ARTIFACTS_PATH}/feature_pipe"


# Loading models from S3
model = mlflow.pyfunc.load_model(MODEL_URI)
feature_pipe = mlflow.sklearn.load_model(FEATURE_PIPE_URI)

# Load data from some data source
data = pd.read_parquet("data/green_tripdata_2021-02.parquet")

# Inference on validation data
X, y = preprocess(data, target=True, filter_target=True)
X = feature_pipe.transform(X)
y_pred = model.predict(X)
print(mean_squared_error(y, y_pred) ** 0.5)

Expected:

In [ ]:
client.get_run(RUN_ID).data.metrics["rmse_valid"]

<br>

**Remark.** MLflow seems to like working with latest versions in the registry by default. This makes sense. But a workaround to getting all models at a given stage is the following (sorted by latest at index 0).

In [ ]:
from datetime import datetime

STAGE = "Archived"
runs = client.search_model_versions(f"name='{MODEL_NAME}'")

print(f"({MODEL_NAME})")
print(f"Stage: {STAGE}")
for run in runs:
    if run.current_stage == STAGE:
        last_updated = run.last_updated_timestamp / 1000    # ms
        print("\n  Version:", run.version)
        print("  Run ID:", run.run_id)
        print("  Status:", run.status)
        print("  Last Modified:", datetime.fromtimestamp(last_updated).strftime('%Y-%m-%d %H:%M:%S'))

(03-mlflow-experiment/appendix/TPE)=
## Appendix: TPE algorithm

Samplers which determine the sequence of trials are specified when creating a Optuna study: 

```python
study = create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler()
)
```


```{figure} ../../img/optuna-samplers.png
---
name: optuna-samplers
---
List of all [sampling algorithms](https://optuna.readthedocs.io/en/stable/reference/samplers.html) as of version 2.10.0.
```

Optuna uses  **Tree-Structured Parzen Estimater** (TPE) {cite}`bergstra` as the default sampler. This algorithm estimates the probability density of each parameter independently: 

> On each trial, for each parameter, TPE fits one Gaussian Mixture Model (GMM) `l(x)` to the set of parameter values associated with the best objective values, and another GMM `g(x)` to the remaining parameter values. It chooses the parameter value `x` that maximizes the ratio `l(x)/g(x)`. &mdash; ([optuna.samplers.TPESampler](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html))

In general, we expect TPE to be more efficient than random search. To demonstrate TPE, we minimize the following objective function:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

%matplotlib inline
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')


def plot_contour(f, title, ax, x_range, y_range, cmap=cm.cividis):
    """Generate 2d contour plot of function f with two parameters."""

    # Plot surface on xy plane; choose 3d or 2d plot
    x = np.linspace(x_range[0], x_range[1], 100)
    y = np.linspace(y_range[0], y_range[1], 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)
    
    # Plot
    ax.contourf(X, Y, Z, cmap=cmap)
    ax.set_xlim(x_range)
    ax.set_ylim(y_range)
    ax.set_title(title)
    ax.set_xlabel(r"$w_1$")
    ax.set_ylabel(r"$w_2$")
    return ax


def plot_surface(f, title, ax, x_range, y_range, cmap=cm.viridis):
    """Generate 3d surface plot {(x, y, f(x, y)) | x ∈ x_range, y ∈ y_range}."""

    # Plot surface on xy plane; choose 3d or 2d plot
    x = np.linspace(x_range[0], x_range[1], 100)
    y = np.linspace(y_range[0], y_range[1], 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)

    # Plot    
    ax.plot_surface(X, Y, Z, cmap=cmap, linewidth=1, color="#000", antialiased=False)
    ax.set_zlabel("loss", rotation=90)
    
    # Formatting plot
    plt.xlim(x_range)
    plt.ylim(y_range)
    plt.title(title)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    plt.tight_layout()

    return ax


def plot_results(study, ax, p1, p2, f, x_range, y_range, figsize=(8, 5)):
    """Scatter plot optimization steps with contour in background."""
    
    plot_contour(f, "objective", ax=ax, x_range=x_range, y_range=y_range)
    study.trials_dataframe().plot(
        kind='scatter', 
        ax=ax,
        figsize=figsize,
        color='C3', edgecolor='black',
        x='params_'+p1, y='params_'+p2,
        xlabel=p1, ylabel=p2
    )
    plt.axis("equal")

In [ ]:
def f(x, y):
    """Flat surface with three holes."""
    r = 4.0
    center = [(0, 0), (3, 3), (6, -3)]
    factor = [-5.0, -4.5, -4.8]
    for j, c in enumerate(center):
        r += factor[j] * np.exp(-(x - c[0])**2 - (y - c[1])**2)
    return r


fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection='3d')
plot_surface(f, title="Objective", x_range=(-4, 10), y_range=(-6, 6), ax=ax);

Note that from the color of the surface that we get better minimas for decreasing $\mathsf x.$

In [ ]:
import optuna

def objective(trial):
    x = trial.suggest_float('x', -4, 10)
    y = trial.suggest_float('y', -6,  6)
    return f(x, y)


min_f = optuna.create_study(direction="minimize")
min_f.optimize(objective, n_trials=50) # TPE default sampler

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1)
plot_results(min_f, ax, 'x', 'y', f, x_range=(-4.5, 10.5), y_range=(-6.5, 6.5), figsize=(4.5, 4))

TPE algorithm (luckily) converged to the global minima. It completely missed out on the other two holes:

In [ ]:
fig = optuna.visualization.plot_contour(min_f, params=["x", "y"])
fig.update_layout(width=800, height=600)
fig.show(renderer="svg")

(03-mlflow-experiment/appendix/AWS-deployment)=
## Appendix: AWS deployment

MLflow uses two components for persisting runs: **backend store** and **artifact store**. The backend store persists run metadata such as  parameters, metrics, and tags. These are best stored in a relational DB allowing queries using SQL. On the other hand, the artifact store persists large files such as serialized models and config files. The backend can be any SQLAlchemy compatible database and the artifact store any remote file storage solutions. Below we deploy an MLflow tracking server remotely in an [EC2](https://aws.amazon.com/ec2/) with PostgreSQL backend using [RDS](https://aws.amazon.com/rds/) and S3 as artifact store.

### EC2 Instance

Launch a t2.micro EC2 instance with name `mlflow-tracking-server` and with Amazon Linux 2 AMI (HVM) 64-bit (x86) OS in an IAM user. See our [previous notebook](https://particle1331.github.io/ok-transformer/nb/mlops/1-intro.html#renting-an-ec2-instance) for more details. Keep the default values for the other settings. Edit inbound rules in security groups:


```{figure} ../../img/inbound-rules.png
---
name: ec2-inbound
width: 100%
---
Our instance should accept incoming SSH (port 22) and HTTP connections (port 5000). Specify CIDR blocks to specify the range of IP addresses that has access to the tracking server. Choosing `0.0.0.0/0` allows all incoming HTTP access.
```

### PostgreSQL DB

Create database in the RDS console:

```{figure} ../../img/rds-template.png
---
width: 100%
---
Choosing an engine and template.
```

```{figure} ../../img/rds-identifier.png
---
width: 100%
---

```

```{figure} ../../img/rds-name.png
---
width: 100%
---
Choosing identifier, and other names. Save master username, password, and initial database name. This will be used later. On the RDS dashboard, click on the database and look at "Connectivity & Security". Take note of the endpoint which you will find here.
```

Select the VPC security group of the DB under the same tab. Click the security group ID, and edit inbound rules by adding a new rule that allows PostgreSQL connections on the port 5432 from the security group of the EC2 instance for the tracking server:

```{figure} ../../img/postgres.png
---
width: 100%
---

Allow postgres connections from the tracking server to the backend database. 
Select the security group of the EC2 instance {numref}`ec2-inbound`. This allows the 
tracking server to connect to the PostgreSQL database. 
```


### Server config and launch

Here we connect to the `mlflow-tracking-server` EC2 instance from our local terminal:

```bash
chmod 400 ~/.ssh/mlflow-tacking.pem
ssh -i "~/.ssh/mlflow-tracking.pem" ec2-user@ec2-34-209-62-152.us-west-2.compute.amazonaws.com
```

Run the following installation steps inside:

```
sudo yum update
pip3 install mlflow boto3 psycopg2-binary
aws configure
aws s3 ls       # Test
```

Running the server: 

```bash
export DB_USER=mlflow
export DB_PASSWORD=ZbTddA0Zc8LxYcdLFUQr
export DB_ENDPOINT=mlflow-backend-database.csegt7oxppl.us-west-2.rds.amazonaws.com
export DB_NAME=mlflow_backend_database
export S3_BUCKET_NAME=mlflow-artifact-store-2

mlflow server -h 0.0.0.0 -p 5000 \
    --backend-store-uri=postgresql://${DB_USER}:${DB_PASSWORD}@${DB_ENDPOINT}:5432/${DB_NAME} \
    --default-artifact-root=s3://${S3_BUCKET_NAME}
```

### Running a remote experiment

Since we are allowed by the inbound rules to send information via HTTP to the tracking server, we should be able to send requests to it. Note that everything that follows is done in our local Jupyter notebook. After setting the tracking URI we should be able to manage experiments and the model registry through the client as before.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

TRACKING_SERVER_HOST = "ec2-34-209-62-152.us-west-2.compute.amazonaws.com"
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")
client = MlflowClient(tracking_uri=f"http://{TRACKING_SERVER_HOST}:5000")

Running an example experiment.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)


mlflow.set_experiment("iris")
X, y = load_iris(return_X_y=True)

for C in [10, 1, 0.1, 0.01, 0.001]:
    with mlflow.start_run(nested=True):
        params = {"C": C, "random_state": 42}
        lr = LogisticRegression(**params).fit(X, y)
        y_pred = lr.predict(X)

        mlflow.log_metric("accuracy", accuracy_score(y, y_pred))
        mlflow.log_params(params)
        mlflow.sklearn.log_model(lr, artifact_path="models")

<br>

```{figure} ../../img/aws-runs.png
---
width: 100%
---

Runs are recorded in the tracking UI.

```


```{figure} ../../img/s3.png
---
width: 100%
---

We can see the runs artifacts stored in the S3 bucket.

```
